# Cold-Started Alpha Sweep

This notebook demonstrates how to perform an angle of attack (alpha) sweep using the Flow360 Python API. The process involves creating a project, defining simulation parameters, launching a series of cases with varying alpha values, and finally generating a comprehensive report summarizing the results.

**Note:** The settings in this example are by no means a validation setup; they are crafted to showcase the capabilities of Flow360 and we have intentionally reduced node count and example FC cost. For rigorous validation, modify the settings as needed.

## 1. Setup and Imports

The following cell imports the necessary libraries and modules. `flow360` is the main package for interacting with the Flow360 platform. Specific modules for defining simulation parameters, creating reports, and handling units are also imported.


In [1]:
import flow360 as fl
from flow360.examples import EVTOL

## 2. Project Creation

A Flow360 `Project` is a container for simulations and their associated assets, such as geometries and meshes. A project can be initiated either from an existing volume mesh or from a CAD geometry file. When starting from a geometry, meshing parameters must be defined to instruct the platform on how to generate the volume mesh.


In [2]:
EVTOL.get_files()

project = fl.Project.from_geometry(EVTOL.geometry, name="Cold-Started Alpha Sweep")

geometry = project.geometry
geometry.group_faces_by_tag("faceName")

[12:04:13] INFO: Geometry successfully submitted:                                                                  
                   type   = Geometry                                                                               
                   name   = sweep_evtol_from_geometry                                                              
                   id     = geo-b66e23df-1030-4070-89e7-3d2e9b86e46d                                               
                   status = uploaded                                                                               
           

           INFO: Waiting for geometry to be processed.

[12:06:07] INFO: Regrouping face entities under `faceName` tag (previous `_color`).

## 3. Meshing parameters

We define some simple meshing parameters to achieve appropriate resolution for this example. The use of `AngleBasedRefinement` will allow us to better model the flow at the leading edge.

In [3]:
with fl.SI_unit_system:
    meshing_params = fl.MeshingParams(
        defaults=fl.MeshingDefaults(
            boundary_layer_first_layer_thickness=1e-5, surface_max_edge_length=1
        ),
        volume_zones=[fl.AutomatedFarfield()],
        refinements=[
            fl.SurfaceEdgeRefinement(
                name="leading_edges",
                edges=[geometry["leadingEdge"]],
                method=fl.AngleBasedRefinement(value=2 * fl.u.deg),
            )
        ],
    )

           INFO: using: SI unit system for unit inference.

## 4. Boundary Conditions

Boundary conditions define the physical behavior at the boundaries of the computational domain. We need to apply appropriate `Wall` and `Freestream` conditions to different surfaces of the geometry.

Surface selection can be done using exact names or with wildcards, for example:
- geometry["*"] will select all geometry boundaries
- geometry["*pylon"] will select all geometry boundaries that end with `pylon`

In [4]:
models = [
    fl.Wall(surfaces=[geometry["*"]]),
    fl.Freestream(surfaces=fl.AutomatedFarfield().farfield),
]

## 5. Simulation Parameters

The `SimulationParams` object encapsulates all settings for a simulation run. This includes meshing parameters, reference geometry, operating conditions, time-stepping scheme, physical models, and output specifications.

In [5]:
with fl.SI_unit_system:
    params = fl.SimulationParams(
        meshing=meshing_params,
        reference_geometry=fl.ReferenceGeometry(moment_center=(0, 0, 0), moment_length=1, area=1),
        operating_condition=fl.AerospaceCondition(velocity_magnitude=100, alpha=0 * fl.u.deg),
        time_stepping=fl.Steady(max_steps=5000, CFL=fl.AdaptiveCFL()),
        models=[
            *models,
            fl.Fluid(
                navier_stokes_solver=fl.NavierStokesSolver(),
                turbulence_model_solver=fl.SpalartAllmaras(),
            ),
        ],
        outputs=[
            fl.VolumeOutput(
                output_format=["tecplot"],
                output_fields=[
                    "Mach",
                    "Cp",
                    "mut",
                    "mutRatio",
                    "primitiveVars",
                    "qcriterion",
                ],
            ),
            fl.SurfaceOutput(
                surfaces=[
                    geometry["fuselage"],
                    geometry["*pylon"],
                    geometry["*wing"],
                    geometry["*tail"],
                ],
                output_fields=[
                    "Cp",
                    "yPlus",
                    "Cf",
                    "CfVec",
                    "primitiveVars",
                    "wallDistance",
                ],
                output_format=["tecplot"],
            ),
        ],
    )

           INFO: using: SI unit system for unit inference.

## 6. Executing an Alpha Sweep

A sweep is performed by iterating through a list of alpha values. In each iteration, the `operating_condition` in `SimulationParams` is updated with a new alpha, and new `Case` is run using `project.run_case()`. The returned `Case` objects are collected in a list for later analysis and report generation. The cases will be running in parallel on the cloud. We then wait for all cases to complete before proceeding.

To not recreate the entire `SimulationParams` every time, we can modify existing one's alpha angle by doing `params.operating_condition.alpha = alpha_angle`.


In [6]:
case_list = []

alphas = [-10, -5, 0, 5, 10, 12, 14] * fl.u.deg

for alpha_angle in alphas:
    params.operating_condition.alpha = alpha_angle

    case = project.run_case(params=params, name=f"alpha_{alpha_angle.value}_case")

    print(f"The case ID is: {case.id} with {alpha_angle=} ")
    case_list.append(case)

print("Waiting for cases to complete...")
for case in case_list:
    case.wait()
print("All cases completed.")

[12:26:25] INFO: using: SI unit system for unit inference.

[12:26:29] INFO: Successfully submitted:                                                                           
                   type   = Case                                                                                   
                   name   = alpha_-10_case                                                                         
                   id     = case-420f149b-69b1-41f1-a8b5-20ee88f153d2                                              
                   status = pending                                                                                
           

The case ID is: case-420f149b-69b1-41f1-a8b5-20ee88f153d2 with alpha_angle=unyt_quantity(-10, 'degree') 


           INFO: using: SI unit system for unit inference.

[12:26:31] INFO: Successfully submitted:                                                                           
                   type   = Case                                                                                   
                   name   = alpha_-5_case                                                                          
                   id     = case-d9f7acb8-7c1a-44b1-8672-b0bbef43859a                                              
                   status = pending                                                                                
           

...

The case ID is: case-e55b6e47-3a45-42f8-854c-0b0d629c250a with alpha_angle=unyt_quantity(12, 'degree') 


           INFO: using: SI unit system for unit inference.

[12:26:46] INFO: Successfully submitted:                                                                           
                   type   = Case                                                                                   
                   name   = alpha_14_case                                                                          
                   id     = case-fa4f47df-9363-4e34-ac66-178c41a0c30b                                              
                   status = pending                                                                                
           

The case ID is: case-fa4f47df-9363-4e34-ac66-178c41a0c30b with alpha_angle=unyt_quantity(14, 'degree') 
Waiting for cases to complete...


All cases completed.
